De acuerdo con la base de datos Car Crash (Apendice A, Conjunto 1: Datos de Accidentes Automovilísticos del Libro Guia Analítica de Negocios Comunicación con Datos), seleccionar los datos para el condado de San Francisco y la Ciudad de San Francisco. Predecir si un accidente automovilístico se provocó en una autopista o no. Para la predicción de los accidentes, es importante tener en cuenta las siguientes variables: Dia de la Semana, Nivel de Impacto (ViolCat), Mes, Cantidad de Luz Diurna, y Cielo Despejado.

In [3]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

#Cargar la base de datos

In [4]:
nxl='/content/1. BD2_CarCrash.xlsx'
DFcompleto_=pd.read_excel(nxl,sheet_name=0) #DataFrame : Tabla de datos - Tablas de Word

# Filtrar por condado y ciudad de San Francisco
DF_sf = DFcompleto_[(DFcompleto_['County'] == 'SAN FRANCISCO') & (DFcompleto_['City'] == 'SAN FRANCISCO')]

# Seleccionar las variables de trabajo
DF = DF_sf[['Weekday','ViolCat', 'ClearWeather', 'Month','CrashType', 'Highway','Daylight']]

# Mostrar las primeras filas del DataFrame filtrado
display(DF.head())

,Weekday,ViolCat,ClearWeather,Month,CrashType,Highway,Daylight
907,7,1,1,3,A,1,1
1580,2,9,1,2,A,0,0
1581,7,3,1,1,A,0,0
1586,7,3,1,3,A,0,0
1587,3,3,1,3,A,0,0


In [5]:
DF = DF.dropna()#Eliminar registros incompletos

In [6]:
from sklearn.preprocessing import LabelEncoder

# Crear un objeto LabelEncoder
codificador = LabelEncoder()

# Ajustar y transformar la columna 'CrashType'
DF['CrashType'] = codificador.fit_transform(DF['CrashType'])

# Mostrar las primeras filas para ver el cambio
display(DF.head(26000))

,Weekday,ViolCat,ClearWeather,Month,CrashType,Highway,Daylight
907,7,1,1,3,0,1,1
1580,2,9,1,2,0,0,0
1581,7,3,1,1,0,0,0
1586,7,3,1,3,0,0,0
1587,3,3,1,3,0,0,0
...,...,...,...,...,...,...,...
112598,5,11,1,8,6,0,0
112599,5,11,1,3,6,0,1
112600,6,10,0,4,6,0,0
112601,4,10,1,4,6,0,0


In [7]:
DF.columns

Index(['Weekday', 'ViolCat', 'ClearWeather', 'Month', 'CrashType', 'Highway',
       'Daylight'],
      dtype='object')

In [8]:
#Separar variables de entrada - variables de salida
DFB = DF[['Weekday','ClearWeather','Month','Daylight','ViolCat','CrashType']]
yd = DF[['Highway']]#Variables de Salida
display(DFB.head())
display(yd.head())

,Weekday,ClearWeather,Month,Daylight,ViolCat,CrashType
907,7,1,3,1,1,0
1580,2,1,2,0,9,0
1581,7,1,1,0,3,0
1586,7,1,3,0,3,0
1587,3,1,3,0,3,0


,Highway
907,1
1580,0
1581,0
1586,0
1587,0


#Entrenar el Modelo de Bayes

In [9]:
# Suppress scientific notation
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [10]:
#Crear el Modelo
mnb=GaussianNB()

#Entrenar el Modelo
mnb.fit(DFB,yd)

# Datos por categoria
ndat=mnb.class_count_
print('Los datos por categoria son:\n',ndat)
cat=mnb.classes_
print('Las categorias son:\n',cat)

#Determinamos las propiedad estadisticas de las variables
u=mnb.theta_ #La media de variables por categoria
print('La Media de variables es:\n:',u)
sigma=np.sqrt(mnb.var_) #La desviacion estandar de variables por categoria
print('La desviacion estandar de variables es:\n:',sigma)
print('Los limites superiores de las variables son:\n',u+sigma)
print('Los limites inferiores de las variables son:\n',u-sigma)

Los datos por categoria son:
 [1580.  501.]
Las categorias son:
 [0 1]
La Media de variables es:
: [[3.88417722 0.86265823 4.35189873 0.66265823 7.45696203 3.21772152]
 [3.9261477  0.79840319 6.69461078 0.62075848 4.1497006  2.27744511]]
La desviacion estandar de variables es:
: [[1.95062542 0.34420782 2.3433645  0.47280262 3.52136997 1.87371962]
 [2.01603723 0.40119266 3.44163641 0.48519831 2.16198902 1.05765448]]
Los limites superiores de las variables son:
 [[ 5.83480263  1.20686605  6.69526323  1.13546084 10.978332    5.09144114]
 [ 5.94218494  1.19959585 10.13624719  1.10595679  6.31168962  3.33509959]]
Los limites inferiores de las variables son:
 [[1.9335518  0.51845041 2.00853424 0.18985561 3.93559205 1.3440019 ]
 [1.91011047 0.39721054 3.25297437 0.13556017 1.98771158 1.21979063]]


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [11]:
ydp=mnb.predict(DFB)
cm=confusion_matrix(yd,ydp)
print('La matriz de confusion es:\n',cm)

La matriz de confusion es:
 [[1365  215]
 [ 224  277]]


In [12]:
cm=confusion_matrix(yd,ydp)
print('La matriz de confusion es:\n',cm)
VN=cm[0,0]
FP=cm[0,1]
FN=cm[1,0]
VP=cm[1,1]
TD=len(DFB) #Total datos

#Exactitud: Muestra el comportamiento general del modelo
Ex=(VP+VN)/TD
print('La exactitud del modelo es:\n',Ex)

#Tasa de error:Porcentaje de equivocacion
ter=(FP+FN)/TD
print('La tasa de error del modelo es:\n',ter)

#Sensibilidad: Como se comporta el modelo frente a los pre aprobados
Se=VP/(VP+FN)
print('La sensibilidad del modelo es:\n',Se)

#Especificidad: Como se comporta el modelo frente a los pre negados
Esp=VN/(VN+FP)
print('La especificidad del modelo es:\n',Esp)

#Precisión: Capacidad del modelo para predecir VP
Pre=VP/(VP+FP)
print('La precision del modelo es:\n',Pre)

#Precision Negativa: Capacidad del modelo para predecir VN
PreN=VN/(VN+FN)
print('La precision negativa del modelo es:\n',PreN)

La matriz de confusion es:
 [[1365  215]
 [ 224  277]]
La exactitud del modelo es:
 0.7890437289764536
La tasa de error del modelo es:
 0.21095627102354636
La sensibilidad del modelo es:
 0.5528942115768463
La especificidad del modelo es:
 0.8639240506329114
La precision del modelo es:
 0.5630081300813008
La precision negativa del modelo es:
 0.8590308370044053


In [13]:
# Contar el número de accidentes en autopistas y otras vías
highway_counts = DF['Highway'].value_counts()

# Calcular el porcentaje de accidentes en autopistas y otras vías
highway_percentages = (highway_counts / len(DF)) * 100

# Imprimir los resultados
print("Porcentaje de accidentes en San Francisco:")
print(f"En autopista: {highway_percentages[1]:.2f}%")
print(f"En otras vías: {highway_percentages[0]:.2f}%")

Porcentaje de accidentes en San Francisco:
En autopista: 24.07%
En otras vías: 75.93%


In [14]:
# Agrupar por la columna 'Highway' y calcular los valores máximos y mínimos para cada variable
summary_stats = DF.groupby('Highway').agg(['max', 'min'])

# Mostrar los resultados
display(summary_stats)

Weekday     ViolCat     ClearWeather     Month     CrashType      \
            max min     max min          max min   max min       max min   
Highway                                                                    
0             7   1      12   1            1   0    12   1         6   0   
1             7   1      11   1            1   0    12   1         6   0   

        Daylight      
             max min  
Highway               
0              1   0  
1              1   0

El modelo Naive Bayes fue evaluado utilizando una matriz de confusión, a partir de la cual se calcularon diversas métricas clave de rendimiento. A continuación, se presentan los resultados:

* La exactitud del modelo fue del 78.9%, lo que indica que casi 8 de cada 10 predicciones fueron correctas.

* La tasa de error fue del 21%, reflejando el porcentaje de casos en los que el modelo falló.

* La sensibilidad (o recall), que mide la capacidad del modelo para detectar correctamente los casos positivos, fue del 55%, lo que evidencia una tendencia a no identificar adecuadamente muchos casos positivos.

* Por otro lado, la especificidad fue del 86%, mostrando una buena capacidad del modelo para reconocer los casos negativos.

* La precisión del modelo fue del 56%, es decir, poco más de la mitad de los casos predichos como positivos fueron realmente positivos.

* Finalmente, la precisión negativa alcanzó un 85.9%, lo cual indica que el modelo tuvo un alto nivel de acierto al predecir casos negativos.


**El modelo es bastante bueno para predecir cuándo un accidente NO es en una autopista (alta especificidad y precisión negativa). Sin embargo, es menos efectivo para predecir correctamente cuándo un accidente SÍ es en una autopista (sensibilidad y precisión más bajas).**

In [17]:
XI=[5,1,1,3,4,0]
ydpi=mnb.predict_proba([XI])
print(f'La probabilidad de que el accidente NO sea en autopista es: {ydpi[0][0]:.2f}')
print(f'La probabilidad de que el accidente SÍ sea en autopista es: {ydpi[0][1]:.2f}')

La probabilidad de que el accidente NO sea en autopista es: 0.77
La probabilidad de que el accidente SÍ sea en autopista es: 0.23


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(
